In [ ]:
# Célula 0 — Tokens, Drive e código do repositório
import os
from google.colab import drive, userdata

try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")  # opcional, acelera downloads
except Exception:
    print("HF_TOKEN não configurado (ok, só fica mais lento).")

drive.mount("/content/drive")

REPO = "/content/experimentos-llm-local"
if not os.path.exists(REPO):
    !git clone https://github.com/Arthur-galvao/experimentos-llm-local.git {REPO}
else:
    !git -C {REPO} pull
%cd {REPO}

# F0 — Setup, Validação da Arquitetura Dory-4B e Baselines

**Notebook fino (`ESPEC_ANTIGRAVITY.md`):** toda a lógica vive em `lib/` e `scripts/`, e cada motor vLLM roda num processo Python separado para liberar 100% da VRAM ao terminar:
1. `pytest tests/test_critical.py` — suíte de testes obrigatórios.
2. `scripts/f0_prepare.py` & `scripts/check_tokenizer.py` — cache do HF em `/root/.cache/huggingface`, validação dos JSONs do modelo (`ensure_model_files`), alinhamento do tokenizer HF vs vLLM (`fix_mistral_regex`) e geração do split fixo `data/lcb_quick.json` (200 problemas).
3. `scripts/f0_validate_dory_hf.py` — validação de `n_recurrent_loops` $\in \{1, 2, 4\}$ e Logit Lens por loop (processo isolado).
4. `scripts/f0_smoke_and_estimate.py` — Smoke Test (`SMOKE=True`, 5 problemas) e estimativa de Compute Units (CU).
5. `scripts/f0_run_baseline.py` — execução dos 4 baselines (`L1`, `L2`, `L4`, `qwen`) em processos separados (lotes de 128 no vLLM com checkpoint por lote) e geração dos gráficos de Pareto.

In [ ]:
# Célula 1 — Hardware e Instalação (vLLM da PR primeiro, sem fixar transformers/safetensors/tokenizers/huggingface_hub)
!nvidia-smi --query-gpu=name,memory.total --format=csv
!VLLM_USE_PRECOMPILED=1 pip install -q "git+https://github.com/vllm-project/vllm.git@refs/pull/60149/head"
!pip install -q accelerate datasets evalplus matplotlib pytest trl peft
!python -c "import transformers, vllm; print('transformers', transformers.__version__, '| vllm', vllm.__version__)"

In [ ]:
# Célula 2 — Testes unitários, preparação do cache/splits e verificação do tokenizer (HF vs vLLM)
!python -m pytest -v tests/test_critical.py
!python scripts/f0_prepare.py
!python scripts/check_tokenizer.py

In [ ]:
# Célula 3 — Validação HF (loops 1/2/4 + Logit Lens) e Smoke Test vLLM + Estimativa de Compute Units
!python scripts/f0_validate_dory_hf.py
!python scripts/f0_smoke_and_estimate.py

In [ ]:
# Célula 4 — Baselines (1 processo por motor vLLM) + Pareto
# Etapa A (recomendada primeiro): SCOPE="quick" -> HE+ e LCB quick p/ L1, L2, L4 e Qwen (~4.7 h, ~42 CU no A100)
# Etapa B (depois de olhar a etapa A): SCOPE="gate" e GATE_TARGETS=["L2"] (~50 CU). L4 no gate custa ~85 CU.
# Estado atual: etapa A e gate L2 feitos, exceto SciCode; o que já está em results.jsonl é pulado.
RUN_FULL_EVAL = True
SCOPE = "gate"                          # "quick" | "gate" | "both"
QUICK_TARGETS = ["L1", "L2", "L4", "qwen"]
GATE_TARGETS = ["L2"]                   # Qwen não tem gate

if RUN_FULL_EVAL:
    targets = QUICK_TARGETS if SCOPE == "quick" else GATE_TARGETS
    for t in targets:
        !python scripts/f0_run_baseline.py --target {t} --scope {SCOPE}
    !python scripts/f0_run_baseline.py --target plot
else:
    print("RUN_FULL_EVAL = False. Revise a estimativa de CU e mude para True.")
